In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = (
    SparkSession.builder
    .appName("RealTimeEcommercePipeline")
    .master("local[*]")
    .config("spark.jars.packages",
        "org.apache.spark:spark-sql-kafka-0-10_2.12:3.3.0")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")

kafka_df = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "ed-kafka:29092")
    .option("subscribe", "ecommerceOrders")
    .option("startingOffsets", "earliest")
    .load()
)

orders_df = kafka_df.select(
    col("value").cast("string").alias("value"),
    col("topic"),
    col("partition"),
    col("offset"),
    col("timestamp")
)

query = (
    orders_df.writeStream
    .format("console")
    .outputMode("append")
    .option("truncate", False)
    .start()
)

query.awaitTermination()

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json
from pyspark.sql.types import *

spark = (
    SparkSession.builder
    .appName("RealTimeEcommercePipeline")
    .master("local[*]")
    .config("spark.jars.packages","org.apache.spark:spark-sql-kafka-0-10_2.12:3.3.0")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")

kafka_df = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "ed-kafka:29092")
    .option("subscribe", "ecommerceOrders")
    .option("startingOffsets", "earliest")
    .load()
)

orders_df = kafka_df.select(
    col("value").cast("string").alias("value"),
    col("topic"),
    col("partition"),
    col("offset"),
    col("timestamp")
)

order_schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("product_id", StringType(), True),
    StructField("product_name", StringType(), True),
    StructField("category", StringType(), True),
    StructField("sub_category", StringType(), True),
    StructField("brand", StringType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("price", DoubleType(), True),
    StructField("total_amount", DoubleType(), True),
    StructField("city", StringType(), True),
    StructField("payment_method", StringType(), True),
    StructField("order_status", StringType(), True),
    StructField("order_time", StringType(), True)
])

parsed_df = orders_df.select(
    from_json(col("value"), order_schema).alias("data"),
    col("topic"),
    col("partition"),
    col("offset"),
    col("timestamp")
)

final_df = parsed_df.select(
    "data.*",
    "topic",
    "partition",
    "offset",
    "timestamp"
)

query = (
    final_df.writeStream
    .format("console")
    .outputMode("append")
    .option("truncate", False)
    .start()
)

query.awaitTermination()

In [ ]:
# Complete Cleaning Pipeline
from pyspark.sql.functions import *
clean_df = final_df.dropna()
clean_df = clean_df.dropDuplicates(
    ["order_id"]
)
clean_df = clean_df.filter(
    col("quantity") > 0
)
clean_df = clean_df.filter(
    col("price") > 0
)
clean_df = clean_df.filter(
    col("total_amount") > 0
)
clean_df = clean_df.withColumn(
    "order_time",
    to_timestamp(
        "order_time",
        "yyyy-MM-dd HH:mm:ss"
    )
)
clean_df = clean_df.filter(
    col("payment_method").isin(
        "UPI",
        "Credit Card",
        "Debit Card",
        "Cash"
    )
)
clean_df = clean_df.filter(
    col("order_status").isin(
        "Placed",
        "Processing",
        "Delivered",
        "Cancelled"
    )
)

clean_df.printSchema()

In [ ]:
# Complete Transformation Code.
from pyspark.sql.functions import *

transformed_df = clean_df.withColumn(
    "discount_percent",
    when(col("total_amount") > 100000, 10)
    .when(col("total_amount") >= 50000, 5)
    .otherwise(0)
)

transformed_df = transformed_df.withColumn(
    "discount_amount",
    round(
        col("total_amount") *
        col("discount_percent") / 100,
        2
    )
)

transformed_df = transformed_df.withColumn(
    "final_amount",
    round(
        col("total_amount") -
        col("discount_amount"),
        2
    )
)

transformed_df = transformed_df.withColumn(
    "processing_time",
    current_timestamp()
)

transformed_df = transformed_df.withColumn(
    "year",
    year("order_time")
).withColumn(
    "month",
    month("order_time")
).withColumn(
    "day",
    dayofmonth("order_time")
).withColumn(
    "hour",
    hour("order_time")
)

transformed_df = transformed_df.withColumn(
    "day_of_week",
    date_format("order_time", "EEEE")
)

transformed_df = transformed_df.withColumn(
    "is_weekend",
    when(dayofweek("order_time").isin(1, 7), "Yes")
    .otherwise("No")
)

transformed_df = transformed_df.withColumn(
    "large_order",
    when(col("final_amount") > 100000, "YES")
    .otherwise("NO")
)

transformed_df = transformed_df.withColumn(
    "order_category",
    when(col("final_amount") < 10000, "Low")
    .when(col("final_amount") < 50000, "Medium")
    .otherwise("High")
)

In [ ]:
# Real Time Aggregation
from pyspark.sql.functions import *
city_sales_df = (
    transformed_df
    .groupBy("city")
    .agg(
        sum("final_amount").alias("total_revenue"),
        count("order_id").alias("total_orders")
    )
)
city_query = (
    city_sales_df.writeStream
    .outputMode("complete")
    .format("console")
    .option("truncate", False)
    .start()
)

category_sales_df = (
    transformed_df
    .groupBy("category")
    .agg(
        sum("final_amount").alias("category_revenue"),
        count("order_id").alias("orders")
    )
)
category_query = (
    category_sales_df.writeStream
    .outputMode("complete")
    .format("console")
    .option("truncate", False)
    .start()
)

payment_df = (
    transformed_df
    .groupBy("payment_method")
    .agg(
        count("*").alias("total_transactions"),
        sum("final_amount").alias("revenue")
    )
)
payment_query = (
    payment_df.writeStream
    .outputMode("complete")
    .format("console")
    .option("truncate", False)
    .start()
)


status_df = (
    transformed_df
    .groupBy("order_status")
    .count()
)
status_query = (
    status_df.writeStream
    .outputMode("complete")
    .format("console")
    .option("truncate", False)
    .start()
)

hourly_df = (
    transformed_df
    .groupBy("hour")
    .agg(
        count("order_id").alias("orders"),
        sum("final_amount").alias("revenue")
    )
)
hourly_query = (
    hourly_df.writeStream
    .outputMode("complete")
    .format("console")
    .option("truncate", False)
    .start()
)

product_df = (
    transformed_df
    .groupBy(
        "product_name"
    )
    .agg(
        sum("quantity").alias("total_quantity"),
        sum("final_amount").alias("revenue")
    )
)
product_query = (
    product_df.writeStream
    .outputMode("complete")
    .format("console")
    .option("truncate", False)
    .start()
)

avg_order_df = (
    transformed_df
    .groupBy()
    .agg(
        avg("final_amount").alias("average_order_value")
    )
)
avg_query = (
    avg_order_df.writeStream
    .outputMode("complete")
    .format("console")
    .option("truncate", False)
    .start()
)



In [ ]:
jdbc_url = "jdbc:mysql://mysql:3306/ecommerce_db"

db_properties = {

    "user": "root",

    "password": "password",

    "driver": "com.mysql.cj.jdbc.Driver"

}

def write_to_mysql(batch_df, batch_id):

    print(f"Writing Batch : {batch_id}")

    (
        batch_df.write
        .mode("append")
        .jdbc(
            url=jdbc_url,
            table="orders",
            properties=db_properties
        )
    )

mysql_query = (

    transformed_df.writeStream

    .foreachBatch(write_to_mysql)

    .outputMode("append")

    .option(
        "checkpointLocation",
        "../checkpoints/orders_checkpoint"
    )

    .start()

)

mysql_query.awaitTermination()



In [ ]:
# Complete Production Code.
import logging
import sys

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(message)s",
    handlers=[
        logging.StreamHandler(sys.stdout),
        logging.FileHandler("ecommerce_pipeline.log")
    ]
)

logger = logging.getLogger("EcommercePipeline")

def write_to_mysql(batch_df, batch_id):
    try:
        logger.info(f"Processing Batch : {batch_id}")
        logger.info(
            f"Records : {batch_df.count()}"
        )

        (
            batch_df.write
            .mode("append")
            .jdbc(
                url=jdbc_url,
                table="orders",
                properties=db_properties
            )
        )

        logger.info(
            f"Batch {batch_id} Completed Successfully"
        )

    except Exception as e:

        logger.error(str(e))
mysql_query = (
    transformed_df
    .writeStream
    .queryName("Orders_Stream")
    .foreachBatch(write_to_mysql)
    .outputMode("append")
    .option(
        "checkpointLocation",
        "../checkpoints/orders_checkpoint"
    )
    .start()
)

print(mysql_query.status)
print(mysql_query.lastProgress)

mysql_query.awaitTermination()

In [ ]:
# Complete Optimization Code.
spark.conf.set(
"spark.sql.shuffle.partitions",
"8"
)
spark.conf.set(
"spark.sql.adaptive.enabled",
"true"
)
optimized_df = (
    transformed_df
    .repartition(8)
)
mysql_query = (
optimized_df
.writeStream
.queryName(
"Optimized_Ecommerce_Stream"
)
.foreachBatch(
write_to_mysql
)
.outputMode(
"append"
)
.trigger(
processingTime="10 seconds"
)
.option(
"checkpointLocation",
"../checkpoints/orders_checkpoint"
)
.start()
)
mysql_query.awaitTermination()